In [0]:
#Always leave this cell, the first time it runs it creates the widgets, the second time it runs it does nothing.
dbutils.widgets.text("target_catalog", "") 
dbutils.widgets.text("target_schema", "") 

In [0]:
# import and register the OpenSky datasource 
from pyspark_datasources import OpenSkyDataSource
spark.dataSource.register(OpenSkyDataSource)

#Fetch the catalog and schema from widget values
catalog = dbutils.widgets.get("target_catalog") 
schema = dbutils.widgets.get("target_schema")

df = spark.readStream.format("opensky").load()

df.writeStream.trigger(availableNow=True).toTable(
    f"{catalog}.{schema}.flights",
    outputMode="append",
    checkpointLocation=f"/Volumes/{catalog}/{schema}/opensky/checkpoint"
)

In [0]:
%sql
    
CREATE OR REPLACE TABLE IDENTIFIER(:target_catalog || '.' || :target_schema || '.flights_stats') AS
  SELECT
    COUNT(*) AS num_events,
    COUNT(DISTINCT icao24) AS unique_aircraft,
    MAX(vertical_rate) AS max_asc_rate,
    MIN(vertical_rate) AS max_desc_rate,
    MAX(velocity) AS max_speed,
    MAX(geo_altitude) AS max_altitude,
    TIMESTAMPDIFF(SECOND, MIN(time_ingest), MAX(time_ingest)) AS observation_duration
  FROM IDENTIFIER(:target_catalog || '.' || :target_schema || '.flights');